# Shallow-water equations: LatentNP vs AttentiveNP

Trains both models on PlanetSWE (The Well) with the same data, settings and seed, and records for each:
**number of parameters**, **training time**, and the validation metrics.
Results are saved to `OUTPUT_DIR/results.json`, with checkpoints in one folder per model.

Data (`swe_data.py`): each task is one trajectory at one time t0, with the sensors' recent readings as
context and 4096 grid points at t0 as targets.
- `x = (relative time, x, y, z)` with relative time in [-1, 0] and the position on the unit sphere;
  the model maps this domain to [-1, 1] internally.
- `y = (h, u_theta, u_phi)`, standardized with training-set statistics; the statistics are saved with the
  results, since predictions must be converted back with them.
- Validation uses a fixed set of tasks, drawn once, so every epoch is compared on the same data.
  Metric: predictive NLL per value (from the context only), plus MSE; RMSE per variable is reported at
  the end in physical units.

## Setup

In [ ]:
import json
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# The folder with base.py, layers.py, LNP.py, ANP.py, loss.py, training.py, swe_data.py:
# by default this notebook's own folder (a batch job can set it with NP_CODE_DIR).
MODEL_CODE_DIR = Path(os.environ.get("NP_CODE_DIR", ".")).resolve()
if str(MODEL_CODE_DIR) in sys.path:
    sys.path.remove(str(MODEL_CODE_DIR))
sys.path.insert(0, str(MODEL_CODE_DIR))  # first, so `import LNP` finds LNP.py

# An old LNP/ package in the same folder as LNP.py would be imported instead of it: catch that.
import importlib.util
_spec = importlib.util.find_spec("LNP")
if _spec is None or not (_spec.origin or "").endswith("LNP.py"):
    _found = _spec and (_spec.origin or list(_spec.submodule_search_locations or []))
    raise ImportError(f"`import LNP` finds {_found}, not LNP.py in {MODEL_CODE_DIR}. "
                      "Put the new model files in their own folder, or rename the old LNP/ package.")

from base import NPConfig
from LNP import LatentNP
from ANP import AttentiveNP
from training import linear_kl_schedule, train
from swe_data import (VARIABLES, X_HIGH, X_LOW, load_sensors, make_train_loader,
                      make_validation_batches, prepare_data)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}" + ("" if device.type == "cuda" else "  (no GPU: training will be very slow)"))

## Settings
The same values as the previous `swe_lnp.py` / `swe_anp.py` scripts.

In [ ]:
# Paths: a batch job can override them with the SWE_DATA_DIR and SWE_OUTPUT_DIR environment variables.
WELL_BASE_PATH = Path(os.environ.get("SWE_DATA_DIR", "data/the_well"))
DATASET_NAME = "planetswe"
DOWNLOAD = False                      # True to download the dataset first
OUTPUT_DIR = Path(os.environ.get("SWE_OUTPUT_DIR", "results_np_comparison"))

BATCH_SIZE = 16
STEPS_PER_EPOCH = 200                 # an "epoch" is this many batches, sampled with replacement
VALIDATION_BATCHES = 20               # fixed validation set: 20 x 16 tasks
NUM_WORKERS = 4                       # if loading is slow or runs out of memory on Windows: 0
SEED = 0

MODEL_SETTINGS = dict(
    r_dim=128, z_dim=128, hidden_dim=128, n_hidden=2, dropout=0.05, activation=nn.ReLU, norm=True,
    use_fourier=True, fourier_dims=None, num_frequencies=32, fourier_scale=1.0, learnable_fourier=True,
    n_decoder_hidden=3,
    output_groups=None,               # [1, 2]: separate decoder branches for h and (u_theta, u_phi)
)
LEARNING_RATE = 5e-4
EPOCHS = 200                          # 200 epochs x 200 steps = 40,000 training steps
WARMUP_EPOCHS = 20                    # KL weight ramps to 1; no LR plateau steps or early stopping before
GRAD_CLIP = 1.0
PRINT_EVERY = 2

RUNS = {
    "LatentNP":    dict(model_class=LatentNP,    plateau_patience=16, early_stopping_patience=32),
    "AttentiveNP": dict(model_class=AttentiveNP, plateau_patience=10, early_stopping_patience=20),
}

## Data

In [ ]:
print(f"Loading '{DATASET_NAME}' from {WELL_BASE_PATH} (download={DOWNLOAD})")
trajectories, coordinates, y_mean, y_std = prepare_data(
    WELL_BASE_PATH, DATASET_NAME, splits=("train", "valid"), download=DOWNLOAD)
for split, data in trajectories.items():
    print(f"{split}: {tuple(data.shape)}  (trajectories, times, grid points, variables)")
for name, m, s in zip(VARIABLES, y_mean, y_std):
    print(f"{name:8s} mean {m:.4g}, std {s:.4g}")

sensors = load_sensors(coordinates)
train_batches = make_train_loader(trajectories["train"], coordinates, sensors, STEPS_PER_EPOCH,
                                  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
val_batches = make_validation_batches(trajectories["valid"], coordinates, sensors, VALIDATION_BATCHES,
                                      batch_size=BATCH_SIZE, seed=SEED)

batch = val_batches[0]
print("Batch:", {key: tuple(value.shape) for key, value in batch.items()})
X_DIM, Y_DIM = batch["x_context"].shape[-1], batch["y_context"].shape[-1]

## Training

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def format_duration(seconds):
    hours, rest = divmod(int(round(seconds)), 3600)
    return f"{hours}:{rest // 60:02d}:{rest % 60:02d}"


def make_config():
    # The context sensors are not necessarily among the targets: the model adds them for the posterior.
    return NPConfig(x_dim=X_DIM, y_dim=Y_DIM, x_low=X_LOW, x_high=X_HIGH,
                    target_includes_context=False, **MODEL_SETTINGS)


def save_results():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "results": results,
        "normalization": {"variables": VARIABLES, "y_mean": y_mean.tolist(), "y_std": y_std.tolist()},
        "settings": {"learning_rate": LEARNING_RATE, "epochs": EPOCHS, "warmup_epochs": WARMUP_EPOCHS,
                     "steps_per_epoch": STEPS_PER_EPOCH, "batch_size": BATCH_SIZE,
                     "validation_batches": VALIDATION_BATCHES, "seed": SEED, "device": str(device)},
    }
    with open(OUTPUT_DIR / "results.json", "w") as f:
        json.dump(payload, f, indent=2, default=str)


results, histories, models = {}, {}, {}

for name, run in RUNS.items():
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    torch.manual_seed(SEED)
    model = run["model_class"](make_config()).to(device)
    n_params = count_parameters(model)
    print(f"Parameters: {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=run["plateau_patience"], min_lr=1e-6)

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    history = train(
        model, train_batches, optimizer, EPOCHS, device,
        val_loader=val_batches, scheduler=scheduler,
        warmup_epochs=WARMUP_EPOCHS, kl_schedule=linear_kl_schedule(WARMUP_EPOCHS),
        grad_clip=GRAD_CLIP, early_stopping_patience=run["early_stopping_patience"],
        checkpoint_dir=OUTPUT_DIR / name, print_every=PRINT_EVERY,
    )
    if device.type == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start

    epochs_run = len(history["train_loss"])
    best_epoch = min(range(WARMUP_EPOCHS, epochs_run), key=lambda e: history["val_nll"][e],
                     default=epochs_run - 1)
    results[name] = {
        "parameters": n_params,
        "training_time_s": round(seconds, 1),
        "training_time": format_duration(seconds),
        "seconds_per_epoch": round(seconds / epochs_run, 2),
        "epochs_run": epochs_run,
        "best_epoch": best_epoch + 1,
        "best_val_nll": history["val_nll"][best_epoch],
        "best_val_mse": history["val_mse"][best_epoch],
        "config": asdict(model.cfg),
    }
    histories[name], models[name] = history, model
    save_results()  # after each model, so a crash in the second run keeps the first
    print(f"{name}: {n_params:,} parameters, trained in {format_duration(seconds)} ({epochs_run} epochs)")

## Results

In [ ]:
header = f"{'Model':<13}{'Parameters':>12}{'Training time':>15}{'s / epoch':>11}{'Epochs':>8}{'Best val NLL':>14}{'Best val MSE':>14}"
print(header)
print("-" * len(header))
for name, r in results.items():
    print(f"{name:<13}{r['parameters']:>12,}{r['training_time']:>15}{r['seconds_per_epoch']:>11.2f}"
          f"{r['epochs_run']:>8}{r['best_val_nll']:>14.4f}{r['best_val_mse']:>14.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    epochs = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(epochs, h["train_loss"], label=name)
    axes[1].plot(epochs, h["val_nll"], label=name)
    axes[2].plot(epochs, h["val_mse"], label=name)
for ax, title in zip(axes, ["Training loss (negative ELBO)", "Validation NLL per value", "Validation MSE (standardized)"]):
    ax.axvline(WARMUP_EPOCHS, color="gray", ls=":", lw=1)
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
axes[2].set_yscale("log")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

### Error per variable, in physical units
RMSE of the predictive mean on the validation set, with the best weights of each model.

In [ ]:
@torch.no_grad()
def rmse_per_variable(model, loader, n_samples=8):
    model.eval()
    squared_error, count = torch.zeros(Y_DIM), 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(batch["x_context"], batch["y_context"], batch["x_target"], num_samples=n_samples)
        mean = out.y.mean.mean(0).cpu() * y_std + y_mean        # back to physical units
        target = batch["y_target"].cpu() * y_std + y_mean
        squared_error += ((mean - target) ** 2).sum(dim=(0, 1))
        count += target.shape[0] * target.shape[1]
    return (squared_error / count).sqrt()

torch.manual_seed(SEED)
print(f"{'Model':<13}" + "".join(f"{v:>12}" for v in VARIABLES))
for name, model in models.items():
    rmse = rmse_per_variable(model, val_batches)
    results[name]["val_rmse"] = dict(zip(VARIABLES, rmse.tolist()))
    print(f"{name:<13}" + "".join(f"{e:>12.4g}" for e in rmse))
print(f"{'(data std)':<13}" + "".join(f"{s:>12.4g}" for s in y_std))
save_results()
print(f"\nSaved to {OUTPUT_DIR / 'results.json'}")